# Waste Image Classification — Paper Replication
**Paper:** *"Towards accurate and efficient waste image classification: A hybrid deep learning and machine learning approach"*
*(Nguyen et al., Ain Shams Engineering Journal, 2026)*

This notebook replicates all three paradigms from the paper:
1. **Classical ML** — handcrafted features (SIFT, GLCM, LBP, etc.) + SVM/XGBoost/LightGBM
2. **Deep Learning** — CustomResNet50 with Focal Loss
3. **Hybrid (best)** — ResNet50 as feature extractor → RF feature selection → ML classifiers

> **GPU recommended.** Enable via *Settings → Accelerator → GPU T4 x2*

---


## Step 1 — Add Datasets
Before running, add these 3 datasets to your notebook via *Add Data*:
- `sumn2u/garbage-classification-v2` → Dataset A (10 classes, 19,762 images)
- `mostafaabla/garbage-classification` → Dataset B (12 classes, 15,150 images)
- `feyzazkefe/trashnet` → Dataset C (6 classes, 2,527 images)

Then run the cell below to confirm paths.

In [ ]:
import os

DATASET_PATHS = {
    # "garbage":   "/kaggle/input/garbage-classification-v2",
    # "household": "/kaggle/input/garbage-classification",
    # "trashnet":  "/kaggle/input/trashnet/dataset-resized",
    "garbage": "/kaggle/input/datasets/sumn2u/garbage-classification-v2/original",
    "household": "/kaggle/input/datasets/mostafaabla/garbage-classification/garbage_classification",
    "trashnet":  "/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized",
}

# Select which dataset to run (change this to 'garbage' or 'household')
ACTIVE_DATASET = "garbage"

dataset_path = DATASET_PATHS[ACTIVE_DATASET]
if os.path.exists(dataset_path):
    classes = sorted([d for d in os.listdir(dataset_path)
                      if os.path.isdir(os.path.join(dataset_path, d))])
    total = sum(len(os.listdir(os.path.join(dataset_path, c))) for c in classes)
    print(f"Dataset : {ACTIVE_DATASET}")
    print(f"Path    : {dataset_path}")
    print(f"Classes : {classes}")
    print(f"Total   : {total} images")
else:
    print(f"Path not found: {dataset_path}")
    print("Please add the dataset via Add Data -> Search Datasets")

## Step 2 — Install dependencies & imports

In [ ]:
import subprocess
subprocess.run(["pip", "install", "scikit-image", "-q"], check=True)

import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as tv_models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, classification_report)
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
import xgboost as xgb
import lightgbm as lgb
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"PyTorch : {torch.__version__}")

## Step 3 — Configuration
All settings in one place — mirrors paper Section 4.2 exactly.

In [ ]:
IMAGE_SIZE   = 400
IMAGE_MEAN   = [0.485, 0.456, 0.406]
IMAGE_STD    = [0.229, 0.224, 0.225]

TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15
TEST_RATIO   = 0.15
RANDOM_SEED  = 42
BATCH_SIZE   = 32
NUM_EPOCHS   = 10

FOCAL_GAMMA  = 2.0
FOCAL_ALPHA  = 0.25

TOP_K_VALUES    = [50, 60, 70, 80, 90, 100]
DEFAULT_TOP_K   = 100

RESULTS_DIR = "/kaggle/working/results"
MODELS_DIR  = "/kaggle/working/models"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)

print("Config loaded.")
print(f"  Image size : {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"  Epochs     : {NUM_EPOCHS}")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Top-K      : {TOP_K_VALUES}")

## Step 4 — Dataset & DataLoaders

In [ ]:
def get_train_transforms():
    return T.Compose([
        T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.2),
        T.RandomRotation(degrees=30),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
        T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
        T.ToTensor(),
        T.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD),
    ])

def get_eval_transforms():
    return T.Compose([
        T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD),
    ])

EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

class WasteDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root      = Path(root)
        self.transform = transform
        self.classes   = sorted([d.name for d in self.root.iterdir()
                                  if d.is_dir() and not d.name.startswith(".")])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.samples   = [
            (str(p), self.class_to_idx[cls])
            for cls in self.classes
            for p in (self.root / cls).iterdir()
            if p.suffix.lower() in EXTENSIONS
        ]

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, label

class _SubsetWithTransform(Dataset):
    def __init__(self, base, transform, indices):
        self.base = base; self.transform = transform; self.indices = indices
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        path, label = self.base.samples[self.indices[idx]]
        img = Image.open(path).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, label

def make_dataloaders(dataset_path, batch_size=BATCH_SIZE, num_workers=2):
    full = WasteDataset(dataset_path)
    n    = len(full)
    n_tr = int(n * TRAIN_RATIO)
    n_va = int(n * VAL_RATIO)
    n_te = n - n_tr - n_va
    g    = torch.Generator().manual_seed(RANDOM_SEED)
    tr_idx, va_idx, te_idx = [], [], []
    tr_ds, va_ds, te_ds = random_split(full, [n_tr, n_va, n_te], generator=g)
    tr_idx = tr_ds.indices; va_idx = va_ds.indices; te_idx = te_ds.indices

    train_loader = DataLoader(_SubsetWithTransform(full, get_train_transforms(), tr_idx),
                               batch_size=batch_size, shuffle=True,  num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(_SubsetWithTransform(full, get_eval_transforms(),  va_idx),
                               batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(_SubsetWithTransform(full, get_eval_transforms(),  te_idx),
                               batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    feat_loader  = DataLoader(_SubsetWithTransform(full, get_eval_transforms(),
                               list(range(len(full)))),
                               batch_size=64, shuffle=False, num_workers=num_workers, pin_memory=True)

    print(f"Classes ({len(full.classes)}): {full.classes}")
    print(f"Train: {n_tr} | Val: {n_va} | Test: {n_te}")
    return train_loader, val_loader, test_loader, feat_loader, full.classes

train_loader, val_loader, test_loader, feat_loader, CLASSES = make_dataloaders(dataset_path)
NUM_CLASSES = len(CLASSES)

## Step 5 — Deep Learning Models
Implements paper Section 3.5: CustomResNet50 with GlobalAveragePooling + Focal Loss.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = gamma; self.alpha = alpha
    def forward(self, logits, targets):
        log_p = F.log_softmax(logits, dim=1)
        p     = torch.exp(log_p)
        log_pt = log_p.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt     = p.gather(1, targets.unsqueeze(1)).squeeze(1)
        return (-self.alpha * (1 - pt) ** self.gamma * log_pt).mean()

class CustomResNet50(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        w = tv_models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        base = tv_models.resnet50(weights=w)
        self.backbone   = nn.Sequential(*list(base.children())[:-1])
        self.classifier = nn.Linear(2048, num_classes)

    def forward_features(self, x):
        return self.backbone(x).flatten(1)

    def forward(self, x):
        return self.classifier(self.forward_features(x))

def build_model(name, num_classes, pretrained=True):
    if name == "custom_resnet50":
        return CustomResNet50(num_classes, pretrained)
    elif name == "resnet50":
        m = tv_models.resnet50(weights=tv_models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, num_classes); return m
    elif name == "resnet101":
        m = tv_models.resnet101(weights=tv_models.ResNet101_Weights.IMAGENET1K_V1 if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, num_classes); return m
    elif name == "efficientnet_v2s":
        m = tv_models.efficientnet_v2_s(weights=tv_models.EfficientNet_V2_S_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes); return m
    elif name == "mobilenet_v3":
        m = tv_models.mobilenet_v3_large(weights=tv_models.MobileNet_V3_Large_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes); return m
    elif name == "densenet121":
        m = tv_models.densenet121(weights=tv_models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes); return m
    raise ValueError(f"Unknown model: {name}")

print("Model definitions ready.")

## Step 6 — Training & Evaluation Utilities

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy":  round(accuracy_score(y_true, y_pred) * 100, 2),
        "precision": round(precision_score(y_true, y_pred, average="macro", zero_division=0) * 100, 2),
        "recall":    round(recall_score(y_true, y_pred, average="macro", zero_division=0) * 100, 2),
        "f1":        round(f1_score(y_true, y_pred, average="macro", zero_division=0) * 100, 2),
    }

def train_model(model, train_loader, val_loader, use_focal=False,
                epochs=NUM_EPOCHS, lr=1e-4, save_path=None):
    model = model.to(DEVICE)
    criterion = FocalLoss(FOCAL_GAMMA, FOCAL_ALPHA) if use_focal else nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_acc, patience_cnt, history = 0.0, 0, []

    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        tr_loss, correct, total = 0.0, 0, 0
        for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(imgs)
            loss   = criterion(logits, labels)
            loss.backward(); optimizer.step()
            tr_loss += loss.item() * imgs.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total   += imgs.size(0)
        scheduler.step()

        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                val_preds.extend(model(imgs.to(DEVICE)).argmax(1).cpu().numpy())
                val_labels.extend(labels.numpy())
        val_acc = accuracy_score(val_labels, val_preds)
        history.append({"epoch": epoch, "val_acc": val_acc})
        print(f"  Epoch {epoch:3d} | val_acc: {val_acc*100:.2f}%")

        if val_acc > best_acc:
            best_acc = val_acc; patience_cnt = 0
            if save_path: torch.save(model.state_dict(), save_path)
        else:
            patience_cnt += 1
            if patience_cnt >= 5:
                print(f"  Early stopping at epoch {epoch}"); break

    train_time = time.time() - t0
    if save_path and os.path.exists(save_path):
        model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    print(f"  Done in {train_time:.1f}s | Best val acc: {best_acc*100:.2f}%")
    return model, history, train_time

@torch.no_grad()
def test_model(model, loader):
    model.eval(); model.to(DEVICE)
    preds, labels = [], []
    for imgs, lbls in loader:
        preds.extend(model(imgs.to(DEVICE)).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    m = compute_metrics(labels, preds)
    print(f"  Test → acc: {m['accuracy']}%  prec: {m['precision']}%  rec: {m['recall']}%  f1: {m['f1']}%")
    return m

@torch.no_grad()
def extract_deep_features(model, loader):
    model.eval(); model.to(DEVICE)
    feats, labels = [], []
    for imgs, lbls in tqdm(loader, desc="Extracting deep features"):
        feats.append(model.forward_features(imgs.to(DEVICE)).cpu().numpy())
        labels.extend(lbls.numpy())
    return np.vstack(feats), np.array(labels)

@torch.no_grad()
def measure_latency(model, loader, n_warmup=5):
    model.eval(); model.to(DEVICE)
    for i, (imgs, _) in enumerate(loader):
        if i >= n_warmup: break
        _ = model(imgs.to(DEVICE))
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    total_ms, total_n = 0.0, 0
    for imgs, _ in loader:
        imgs = imgs.to(DEVICE)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        t = time.perf_counter()
        _ = model(imgs)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        total_ms += (time.perf_counter() - t) * 1000
        total_n  += imgs.size(0)
    return round(total_ms / total_n, 3)

print("Training utilities ready.")

## Step 7 — Handcrafted Feature Extraction
Implements paper Section 3.2: 1305-d feature vector per image.

In [ ]:
def grabcut_segment(img_bgr):
    h, w   = img_bgr.shape[:2]
    mask   = np.zeros((h, w), np.uint8)
    bgd    = np.zeros((1, 65), np.float64)
    fgd    = np.zeros((1, 65), np.float64)
    margin = max(10, int(min(h, w) * 0.10))
    rect   = (margin, margin, w - 2*margin, h - 2*margin)
    try:
        cv2.grabCut(img_bgr, mask, rect, bgd, fgd, 5, cv2.GC_INIT_WITH_RECT)
        bm = np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 255, 0).astype(np.uint8)
    except: bm = np.ones((h, w), np.uint8) * 255
    cnts, _ = cv2.findContours(bm, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if cnts:
        refined = np.zeros_like(bm)
        cv2.drawContours(refined, [max(cnts, key=cv2.contourArea)], -1, 255, -1)
        return refined
    return bm

def color_basic(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
    f = []
    for c in range(3):
        ch = hsv[:,:,c].flatten(); ch = ch if len(ch) else np.zeros(1)
        m = np.mean(ch); s = np.std(ch)+1e-7
        hist, _ = np.histogram(ch, bins=32, density=True); hist = hist[hist>0]
        f += [m, s, float(np.mean(((ch-m)/s)**3)), float(np.mean(((ch-m)/s)**4)),
              -np.sum(hist*np.log2(hist+1e-9))]
    return np.array(f, np.float32)   # 15

def color_hist(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    h1  = cv2.normalize(cv2.calcHist([hsv], [0,1,2], None, [8,8,8], [0,180,0,256,0,256]), None).flatten()
    h2  = cv2.normalize(cv2.calcHist([img],  [0,1,2], None, [8,8,8], [0,256,0,256,0,256]), None).flatten()
    return np.concatenate([h1, h2]).astype(np.float32)   # 1024

def contour_feats(mask):
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return np.zeros(5, np.float32)
    c  = max(cnts, key=cv2.contourArea)
    a  = cv2.contourArea(c); p = cv2.arcLength(c, True)
    x,y,w,h = cv2.boundingRect(c)
    sol = a / (cv2.contourArea(cv2.convexHull(c)) + 1e-7)
    return np.array([a, p, w/(h+1e-7), a/(w*h+1e-7), sol], np.float32)   # 5

def hu_feats(gray):
    hu = cv2.HuMoments(cv2.moments(gray)).flatten()
    return (-np.sign(hu) * np.log10(np.abs(hu)+1e-10)).astype(np.float32)   # 7

def glcm_feats(gray):
    g = np.clip(gray, 0, 255).astype(np.uint8) // 16
    glcm = graycomatrix(g, [1], [0,np.pi/4,np.pi/2,3*np.pi/4], 16, symmetric=True, normed=True)
    f = []
    for p in ["contrast","dissimilarity","homogeneity","energy","correlation"]:
        f.extend(graycoprops(glcm, p)[0].tolist())
    return np.array(f, np.float32)   # 20

def lbp_feats(gray):
    g    = np.clip(gray, 0, 255).astype(np.uint8)
    lbp  = local_binary_pattern(g, 8, 1, method="uniform")
    hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0,10), density=True)
    return hist.astype(np.float32)   # 10

def orb_feats(gray):
    g = np.clip(gray, 0, 255).astype(np.uint8)
    _, d = cv2.ORB_create(500).detectAndCompute(g, None)
    return d.mean(0).astype(np.float32) if d is not None and len(d) else np.zeros(32, np.float32)   # 32

def sift_feats(gray):
    g = np.clip(gray, 0, 255).astype(np.uint8)
    _, d = cv2.SIFT_create().detectAndCompute(g, None)
    return d.mean(0).astype(np.float32) if d is not None and len(d) else np.zeros(128, np.float32)   # 128

def gist_feats(img, grid=4, n_ori=4):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)
    h, w = gray.shape; ch, cw = h//grid, w//grid; f = []
    for t in range(n_ori):
        theta   = t * np.pi / n_ori
        kernel  = cv2.getGaborKernel((21,21), 4.0, theta, 10.0, 0.5, 0)
        filt    = cv2.filter2D(gray, cv2.CV_32F, kernel)
        for i in range(grid):
            for j in range(grid):
                f.append(float(np.mean(np.abs(filt[i*ch:(i+1)*ch, j*cw:(j+1)*cw]))))
    return np.array(f, np.float32)   # 64

def extract_handcrafted(pil_img, use_seg=True):
    """Returns 1305-d feature vector for one PIL image."""
    img  = cv2.cvtColor(np.array(pil_img.convert("RGB")), cv2.COLOR_RGB2BGR)
    img  = cv2.resize(img, (400, 400))
    mask = grabcut_segment(img) if use_seg else np.ones((400,400), np.uint8)*255
    masked  = img.copy(); masked[mask==0] = 0
    gray    = cv2.cvtColor(masked, cv2.COLOR_BGR2GRAY).astype(np.float32)
    feats   = np.concatenate([
        color_basic(masked),    # 15
        color_hist(masked),     # 1024
        contour_feats(mask),    # 5
        hu_feats(gray),         # 7
        glcm_feats(gray),       # 20
        lbp_feats(gray),        # 10
        orb_feats(gray),        # 32
        sift_feats(gray),       # 128
        gist_feats(masked),     # 64
    ])
    return np.nan_to_num(feats, 0.0).astype(np.float32)   # 1305

def extract_all_handcrafted(dataset_path, use_seg=True):
    """Extract handcrafted features from entire dataset. Returns X, y, classes."""
    root    = Path(dataset_path)
    classes = sorted([d.name for d in root.iterdir() if d.is_dir()])
    c2i     = {c:i for i,c in enumerate(classes)}
    samples = [(str(p), c2i[c]) for c in classes
                for p in (root/c).iterdir() if p.suffix.lower() in EXTENSIONS]
    X, y = [], []
    for path, label in tqdm(samples, desc="Extracting handcrafted features"):
        try:
            X.append(extract_handcrafted(Image.open(path), use_seg))
            y.append(label)
        except: pass
    return np.array(X, np.float32), np.array(y, np.int64), classes

print("Handcrafted feature functions ready.")

## Step 8 — ML Classifiers & Feature Selection
Implements paper Sections 3.3–3.4.

In [ ]:
def get_classifiers():
    return {
        "LoR":      LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs", random_state=42),
        "KNN":      KNeighborsClassifier(n_neighbors=5),
        "SVM":      SVC(kernel="rbf", C=10.0, gamma="scale", random_state=42),
        "DT":       DecisionTreeClassifier(random_state=42),
        "RF":       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        "XGBoost":  xgb.XGBClassifier(n_estimators=200, learning_rate=0.1, random_state=42, n_jobs=-1, verbosity=0),
        "LightGBM": lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1),
    }

class RFFeatureSelector:
    def __init__(self, top_k=100):
        self.top_k = top_k
        self.rf    = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        self.idx_  = None
    def fit(self, X, y):
        self.rf.fit(X, y)
        self.idx_ = np.argsort(self.rf.feature_importances_)[::-1][:self.top_k]
        return self
    def transform(self, X): return X[:, self.idx_]
    def fit_transform(self, X, y): return self.fit(X, y).transform(X)

def run_classifiers(X_train, y_train, X_test, y_test, label=""):
    scaler  = StandardScaler()
    X_tr    = scaler.fit_transform(X_train)
    X_te    = scaler.transform(X_test)
    rows    = []
    for name, clf in get_classifiers().items():
        t0  = time.time()
        clf.fit(X_tr, y_train)
        m   = compute_metrics(y_test, clf.predict(X_te))
        m["model"] = name; m["train_s"] = round(time.time()-t0, 2)
        rows.append(m)
    df = pd.DataFrame(rows).set_index("model")[["accuracy","precision","recall","f1","train_s"]]
    print(f"\n{label}")
    print(df.to_string())
    return df

def run_hybrid(X_deep_train, y_train, X_deep_test, y_test,
               top_k=DEFAULT_TOP_K, label="Hybrid"):
    """2048-d deep features -> RF feature selection -> ML classifiers."""
    scaler   = StandardScaler()
    X_tr     = scaler.fit_transform(X_deep_train)
    X_te     = scaler.transform(X_deep_test)
    sel      = RFFeatureSelector(top_k=top_k)
    X_tr_sel = sel.fit_transform(X_tr, y_train)
    X_te_sel = sel.transform(X_te)
    print(f"  Feature selection: 2048d -> top-{sel.top_k}d")
    return run_classifiers(X_tr_sel, y_train, X_te_sel, y_test, label=label)

print("ML classifiers ready.")

## Step 9 — Train CustomResNet50 (DL Paradigm)
This is the backbone for the hybrid pipeline. Takes ~20–40 min on T4 GPU for 30 epochs.

In [ ]:
backbone_path = os.path.join(MODELS_DIR, f"custom_resnet50_{ACTIVE_DATASET}.pt")

model = build_model("custom_resnet50", NUM_CLASSES, pretrained=True)
print(f"Training CustomResNet50 on {ACTIVE_DATASET} ({NUM_CLASSES} classes)...")
model, history, train_time = train_model(
    model, train_loader, val_loader,
    use_focal=True,
    epochs=NUM_EPOCHS,
    save_path=backbone_path
)
print(f"Training time: {train_time:.1f}s")

val_accs = [h["val_acc"]*100 for h in history]
plt.figure(figsize=(8, 3))
plt.plot(val_accs, color="#1D9E75", linewidth=2)
plt.xlabel("Epoch"); plt.ylabel("Val Accuracy (%)")
plt.title(f"CustomResNet50 — {ACTIVE_DATASET}")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print("\nTest set results:")
dl_metrics = test_model(model, test_loader)
latency    = measure_latency(model, test_loader)
print(f"  Inference latency: {latency} ms/sample")

Grad-CAM and activation of the second Deep Learning model (EfficientNetV2-S)

Add EfficientNetV2-S as second DL model (1 line change in the training cell)
Run all 3 datasets and collect results into one comparison table
Add Grad-CAM on a few sample images

In [ ]:
import subprocess
subprocess.run(["pip", "install", "grad-cam", "-q"], check=True)

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

target_layers = [model.backbone[7][-1]]
mean = np.array(IMAGE_MEAN); std = np.array(IMAGE_STD)

def denorm(img_tensor):
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)
    return np.clip(img * std + mean, 0, 1).astype(np.float32)

samples = {}
for imgs, labels in test_loader:
    for img, lbl in zip(imgs, labels):
        c = int(lbl)
        if c not in samples:
            samples[c] = img
    if len(samples) == len(CLASSES):
        break

cam = GradCAM(model=model, target_layers=target_layers)
model.eval()

items = sorted(samples.items())
n = len(items)
cols = min(n, 6)
blocks = int(np.ceil(n / cols))
rows = blocks * 2

fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.4, rows * 2.4))
axes = np.array(axes).reshape(rows, cols)

for i, (true_cls, img) in enumerate(items):
    col = i % cols
    r_orig, r_over = (i // cols) * 2, (i // cols) * 2 + 1
    input_tensor = img.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = model(input_tensor).argmax(1).item()
    grayscale_cam = cam(input_tensor=input_tensor,
                        targets=[ClassifierOutputTarget(pred)])[0]
    rgb = denorm(img)
    overlay = show_cam_on_image(rgb, grayscale_cam, use_rgb=True)
    axes[r_orig, col].imshow(rgb)
    axes[r_orig, col].set_title(f"true: {CLASSES[true_cls]}", fontsize=9)
    axes[r_orig, col].axis("off")
    axes[r_over, col].imshow(overlay)
    color = "green" if pred == true_cls else "red"
    axes[r_over, col].set_title(f"pred: {CLASSES[pred]}", fontsize=9, color=color)
    axes[r_over, col].axis("off")

for j in range(n, cols * blocks):
    col = j % cols
    axes[(j // cols) * 2, col].axis("off")
    axes[(j // cols) * 2 + 1, col].axis("off")

plt.tight_layout()
cam_path = os.path.join(RESULTS_DIR, f"gradcam_{ACTIVE_DATASET}.png")
plt.savefig(cam_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Grad-CAM figure saved -> {cam_path}")

import gc
cam.__del__()
del cam
gc.collect()
torch.cuda.empty_cache()
model.cpu()
torch.cuda.empty_cache()
print("GPU memory freed after GradCAM.")

second DL model (EfficientNetV2-S)

In [ ]:
effnet_path = os.path.join(MODELS_DIR, f"efficientnet_v2s_{ACTIVE_DATASET}.pt")

model2 = build_model("efficientnet_v2s", NUM_CLASSES, pretrained=True)
print(f"Training EfficientNetV2-S on {ACTIVE_DATASET} ({NUM_CLASSES} classes)...")
train_loader_eff = DataLoader(train_loader.dataset, batch_size=8,
                              shuffle=True,  num_workers=2, pin_memory=True)
val_loader_eff   = DataLoader(val_loader.dataset,   batch_size=8,
                              shuffle=False, num_workers=2)
model2, history2, train_time2 = train_model(
    model2, train_loader_eff, val_loader_eff,
    use_focal=True,
    epochs=NUM_EPOCHS,
    save_path=effnet_path
)

print("\nEfficientNetV2-S test results:")
dl2_metrics = test_model(model2, test_loader)
latency2 = measure_latency(model2, test_loader)
print(f"  Inference latency: {latency2} ms/sample")

df_dl_compare = pd.DataFrame([
    {"model": "CustomResNet50", **dl_metrics, "latency_ms": latency},
    {"model": "EfficientNetV2-S", **dl2_metrics, "latency_ms": latency2},
]).set_index("model")
print("\nDL model comparison:")
print(df_dl_compare.to_string())

dl_compare_path = os.path.join(RESULTS_DIR, f"dl_comparison_{ACTIVE_DATASET}.csv")
df_dl_compare.to_csv(dl_compare_path)
print(f"Saved -> {dl_compare_path}")

del model2
torch.cuda.empty_cache()
print("EfficientNetV2-S freed from GPU.")

## Step 10 — Extract Deep Features
Extract 2048-d GAP vectors from the trained backbone for the hybrid pipeline.

In [ ]:
model.to(DEVICE)
torch.cuda.empty_cache()
print("Extracting 2048-d GAP features from full dataset...")
t_fe = time.time()
X_deep, y_deep = extract_deep_features(model, feat_loader)
fe_time = time.time() - t_fe
print(f"Features shape : {X_deep.shape}")
print(f"Labels shape   : {y_deep.shape}")
print(f"Extraction time: {fe_time:.2f}s")

X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(
    X_deep, y_deep, test_size=TEST_RATIO, random_state=RANDOM_SEED, stratify=y_deep
)
print(f"\nSplit -> train: {len(y_tr_d)} | test: {len(y_te_d)}")

## Step 11 — Hybrid Pipeline (Paper's Best Method)
Replicates Tables 9, 10, 11 from the paper.

In [ ]:
print(f"Running hybrid pipeline on {ACTIVE_DATASET}...")
df_hybrid = run_hybrid(X_tr_d, y_tr_d, X_te_d, y_te_d,
                        top_k=DEFAULT_TOP_K,
                        label=f"Hybrid Results — {ACTIVE_DATASET} (top-{DEFAULT_TOP_K})")

print("\nFeature selection sweep...")
sweep_results = []
for k in TOP_K_VALUES:
    scaler  = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_tr_d)
    X_te_s  = scaler.transform(X_te_d)
    sel     = RFFeatureSelector(top_k=k)
    X_tr_sel = sel.fit_transform(X_tr_s, y_tr_d)
    X_te_sel = sel.transform(X_te_s)
    for clf_name, clf in [("LoR", LogisticRegression(max_iter=1000, random_state=42)),
                            ("SVM", SVC(kernel="rbf", C=10.0, random_state=42))]:
        clf.fit(X_tr_sel, y_tr_d)
        acc = accuracy_score(y_te_d, clf.predict(X_te_sel)) * 100
        sweep_results.append({"top_k": k, "classifier": clf_name, "accuracy": round(acc, 2)})

df_sweep = pd.DataFrame(sweep_results)
print(df_sweep.pivot(index="top_k", columns="classifier", values="accuracy").to_string())

fig, ax = plt.subplots(figsize=(8, 4))
for clf_name in ["LoR", "SVM"]:
    sub = df_sweep[df_sweep.classifier == clf_name]
    ax.plot(sub.top_k, sub.accuracy, marker="o", label=clf_name, linewidth=2)
ax.set_xlabel("Number of selected features (top-K)")
ax.set_ylabel("Test accuracy (%)")
ax.set_title(f"Feature selection sweep — {ACTIVE_DATASET}")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

save the .pt model for testing/predicting with different dataset

In [ ]:
import joblib

best_clf_name = df_hybrid["accuracy"].idxmax()

scaler   = StandardScaler()
X_tr_s   = scaler.fit_transform(X_tr_d)
selector = RFFeatureSelector(top_k=DEFAULT_TOP_K)
X_tr_sel = selector.fit_transform(X_tr_s, y_tr_d)
clf      = get_classifiers()[best_clf_name]
clf.fit(X_tr_sel, y_tr_d)

bundle = {"scaler": scaler, "selector": selector, "classifier": clf,
          "classes": CLASSES, "clf_name": best_clf_name}
bundle_path = os.path.join(MODELS_DIR, f"hybrid_pipeline_{ACTIVE_DATASET}.joblib")
joblib.dump(bundle, bundle_path)
print(f"Saved hybrid pipeline ({best_clf_name}) -> {bundle_path}")

## Step 12 — 5-Fold CV (TrashNet / Table 11)
The paper uses stratified 5-fold CV for TrashNet due to its small size.

In [ ]:
if ACTIVE_DATASET == "trashnet":
    print("Running 5-fold stratified cross-validation...")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    cv_acc = {k: [] for k in ["LoR","KNN","SVM","DT","RF","XGBoost","LightGBM"]}

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X_deep, y_deep)):
        print(f"  Fold {fold+1}/5")
        X_tr_f, X_te_f = X_deep[tr_idx], X_deep[te_idx]
        y_tr_f, y_te_f = y_deep[tr_idx], y_deep[te_idx]
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr_f)
        X_te_s = scaler.transform(X_te_f)
        sel    = RFFeatureSelector(top_k=DEFAULT_TOP_K)
        X_tr_s = sel.fit_transform(X_tr_s, y_tr_f)
        X_te_s = sel.transform(X_te_s)
        for name, clf in get_classifiers().items():
            clf.fit(X_tr_s, y_tr_f)
            cv_acc[name].append(accuracy_score(y_te_f, clf.predict(X_te_s)) * 100)

    print("\n5-Fold CV Results (mean ± std):")
    rows = [{"model": k, "mean_acc": round(np.mean(v), 2), "std_acc": round(np.std(v), 2)}
            for k, v in cv_acc.items()]
    df_cv = pd.DataFrame(rows).set_index("model")
    print(df_cv.to_string())
else:
    print(f"Skipping 5-fold CV (only for trashnet, current: {ACTIVE_DATASET})")

## Step 13 — Comparison Plot
Replicates Figure 5 from the paper: best accuracy per paradigm side by side.

In [ ]:
RUN_ML_PARADIGM = True

if RUN_ML_PARADIGM:
    print("Extracting handcrafted features (this takes a while)...")
    X_hc, y_hc, _ = extract_all_handcrafted(dataset_path, use_seg=False)
    X_tr_hc, X_te_hc, y_tr_hc, y_te_hc = train_test_split(
        X_hc, y_hc, test_size=TEST_RATIO, random_state=RANDOM_SEED, stratify=y_hc
    )
    df_ml = run_classifiers(X_tr_hc, y_tr_hc, X_te_hc, y_te_hc,
                            label="Classical ML Results (handcrafted features)")
else:
    df_ml = None

summary_rows = []
if df_ml is not None:
    b = df_ml["accuracy"].idxmax(); r = df_ml.loc[b]
    summary_rows.append({"paradigm": "Classical ML", "best_model": b,
                         "accuracy": r["accuracy"], "precision": r["precision"],
                         "recall": r["recall"], "f1": r["f1"]})
for name in df_dl_compare.index:
    r = df_dl_compare.loc[name]
    summary_rows.append({"paradigm": "Deep Learning", "best_model": name,
                         "accuracy": r["accuracy"], "precision": r["precision"],
                         "recall": r["recall"], "f1": r["f1"]})
b = df_hybrid["accuracy"].idxmax(); r = df_hybrid.loc[b]
summary_rows.append({"paradigm": "Hybrid (DL + ML)", "best_model": b,
                     "accuracy": r["accuracy"], "precision": r["precision"],
                     "recall": r["recall"], "f1": r["f1"]})

df_paradigm = pd.DataFrame(summary_rows)
print("\n3-paradigm comparison:")
print(df_paradigm.to_string(index=False))

best_ml_acc     = df_ml["accuracy"].max() if df_ml is not None else None
best_dl_name    = df_dl_compare["accuracy"].idxmax()
best_dl_acc     = df_dl_compare["accuracy"].max()
best_hybrid_acc = df_hybrid["accuracy"].max()

paradigms = [
    "Classical ML\n(handcrafted)",
    f"End-to-end DL\n({best_dl_name})",
    "Hybrid\n(ResNet50 + ML)",
]
accuracies  = [best_ml_acc if best_ml_acc else 0, best_dl_acc, best_hybrid_acc]
colors      = ["#B5D4F4", "#1D9E75", "#0F6E56"]
labels_show = [f"{a:.2f}%" if a else "N/A" for a in accuracies]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(paradigms, [a if a else 0 for a in accuracies],
              color=colors, width=0.5, edgecolor="none")
for bar, lbl in zip(bars, labels_show):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            lbl, ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.set_ylabel("Test Accuracy (%)")
ax.set_title(f"Paradigm Comparison - {ACTIVE_DATASET}")
ax.set_ylim([max(0, min(a for a in accuracies if a) - 5), 101])
ax.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

print(f"\nSummary for {ACTIVE_DATASET}:")
if best_ml_acc: print(f"  Classical ML  : {best_ml_acc}%")
print(f"  Deep Learning : {best_dl_acc}%  ({best_dl_name})")
print(f"  Hybrid (best) : {best_hybrid_acc}%")

## Step 14 — Save Results

In [ ]:
result_path = os.path.join(RESULTS_DIR, f"hybrid_{ACTIVE_DATASET}.csv")
df_hybrid.to_csv(result_path)
print(f"Hybrid saved      -> {result_path}")

if RUN_ML_PARADIGM and df_ml is not None:
    ml_path = os.path.join(RESULTS_DIR, f"ml_{ACTIVE_DATASET}.csv")
    df_ml.to_csv(ml_path)
    print(f"ML-only saved     -> {ml_path}")

dl_path = os.path.join(RESULTS_DIR, f"dl_{ACTIVE_DATASET}.csv")
df_dl_compare.to_csv(dl_path)
print(f"DL-only saved     -> {dl_path}")

cmp_path = os.path.join(RESULTS_DIR, f"comparison_{ACTIVE_DATASET}.csv")
df_paradigm.to_csv(cmp_path, index=False)
print(f"Comparison saved  -> {cmp_path}")

sweep_path = os.path.join(RESULTS_DIR, f"feature_sweep_{ACTIVE_DATASET}.csv")
df_sweep.to_csv(sweep_path, index=False)
print(f"Sweep saved       -> {sweep_path}")

print(f"Backbone saved    -> {backbone_path}")
print("\nAll done! Check /kaggle/working/results/ for output files.")

## Next Steps — Your Research Ideas

Now that the baseline is running, here's how to extend each idea in this same notebook:

**Idea #1 — Cross-dataset generalization:**
```python
# Load model trained on 'garbage', test on 'trashnet' without retraining
model_cross = build_model("custom_resnet50", num_classes=6, pretrained=False)
model_cross.load_state_dict(torch.load(".../custom_resnet50_garbage.pt"))
# Run extract_deep_features() on trashnet → run_hybrid()
```

**Idea #7 — XAI / Grad-CAM:**
```python
# pip install grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
target_layers = [model.backbone[-1][-1]]   # last ResNet layer
cam = GradCAM(model=model, target_layers=target_layers)
```

**Idea #5 — Feature fusion:**
```python
# Concatenate handcrafted (1305-d) + deep (2048-d) → 3353-d
X_fused = np.concatenate([X_hc, X_deep], axis=1)
run_classifiers(X_fused[:split], y[:split], X_fused[split:], y[split:])
```

**Idea #4 — Detection pipeline:**
```python
# pip install ultralytics
from ultralytics import YOLO
yolo = YOLO("yolov8s.pt")
# detect → crop → pass to hybrid classifier
```
